---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # CUSTOMER_360 (converted from Dataiku DSS)

    | DSS | Lakeflow Designer |
    |---|---|
    | Project variables min_order_date, analysis_end, session_gap_minutes | `env_config` |
    | compute_recent_orders (SQL recipe, PostgreSQL) | `recent_orders` (SQL, rewritten for Databricks SQL, `${var}` -> env_config) |
    | compute_sessions (Python recipe, pandas) | `sess_gaps`, `sess_numbered`, `sess_flags`, `sessions`, `sessions_duration` (visual + SQL windows: no Python needed) |
    | compute_sessions_by_customer (Group) | `sessions_by_customer` |
    | compute_event_pivot (Pivot, count) | `event_pivot` + `event_pivot_names` |
    | compute_customers_prepared (Prepare) | `customers_prepared` + `customers_drop_x` |
    | compute_customer_360_joined (Join, 3 x LEFT) | `j_orders`, `j_sessions`, `j_events` |
    | compute_customer_360_scored (Prepare) | `j_env` (cross join variables) + `scored` |
    | compute_customer_360 (Window rank) | `rank_in_country` (SQL) |

    **MANUAL (Lakeflow Jobs)**: scenario `DAILY_REFRESH` = job schedule 06:30 Europe/Berlin; `check_dataset` recordCount >= 300 = SQL alert or a validation task; mail reporter on failure = job failure notification to crm-analytics@retail.example.
---

In [0]:
"""
id: "env_config"
template: "python"
templateVersion: "1.0.0"
name: "env_config"
position:
  x: 0
  y: 360
description:
  text: "Project variables."
  hash: ""
previewMode: "1000"
config:
  code: |-
    # DSS project variables (variables.json -> standard)
    result = spark.createDataFrame(
        [("2025-07-01", "2025-11-15", 30)],
        ["min_order_date", "analysis_end", "session_gap_minutes"],
    )
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    # DSS project variables (variables.json -> standard)
    result = spark.createDataFrame(
        [("2025-07-01", "2025-11-15", 30)],
        ["min_order_date", "analysis_end", "session_gap_minutes"],
    )

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [],
    "data__sources": []
}
out = run(config, inputs, spark)
ctx["env_config.result"] = out["result"]


In [0]:
"""
id: "src_orders"
template: "source"
templateVersion: "1.0.0"
name: "src_orders"
position:
  x: 0
  y: 515
description:
  text: "Dataset orders."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_orders"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_orders.data"] = out["data"]


In [0]:
"""
id: "src_order_lines"
template: "source"
templateVersion: "1.0.0"
name: "src_order_lines"
position:
  x: 0
  y: 670
description:
  text: "Dataset order_lines."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_order_lines"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_order_lines"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_order_lines.data"] = out["data"]


In [0]:
"""
id: "src_products"
template: "source"
templateVersion: "1.0.0"
name: "src_products"
position:
  x: 0
  y: 825
description:
  text: "Dataset products."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_products"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_products"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_products.data"] = out["data"]


In [0]:
"""
id: "src_customers"
template: "source"
templateVersion: "1.0.0"
name: "src_customers"
position:
  x: 0
  y: 980
description:
  text: "Dataset customers."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_customers"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_customers"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_customers.data"] = out["data"]


In [0]:
"""
id: "src_web_events"
template: "source"
templateVersion: "1.0.0"
name: "src_web_events"
position:
  x: 0
  y: 1135
description:
  text: "Dataset web_events."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_web_events"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_web_events"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_web_events.data"] = out["data"]


In [0]:
"""
id: "recent_orders"
template: "sql"
templateVersion: "1.0.0"
name: "recent_orders"
position:
  x: 300
  y: 360
description:
  text: "SQL recipe compute_recent_orders: PostgreSQL ::numeric -> DECIMAL (exact sum before ROUND), quoted erp. schema removed; ${min_order_date} from env_config."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT o.customer_id,
           COUNT(DISTINCT o.order_id) AS orders_6m,
           ROUND(SUM(CAST(l.quantity AS DECIMAL(10,0)) * CAST(COALESCE(l.unit_price, p.list_price) AS DECIMAL(12,2)) * (1 - CAST(l.discount_pct AS DECIMAL(5,2)))), 2) AS revenue_6m,
           MAX(o.order_date) AS last_order_date
    FROM src_orders o
    JOIN src_order_lines l ON l.order_id = o.order_id
    LEFT JOIN src_products p ON p.product_id = l.product_id
    CROSS JOIN env_config v
    WHERE o.status = 'COMPLETED'
      AND o.order_date >= CAST(v.min_order_date AS DATE)
    GROUP BY o.customer_id
input:
  - node: "src_orders"
    input_port: "data"
    output_port: "data"
  - node: "src_order_lines"
    input_port: "data"
    output_port: "data"
  - node: "src_products"
    input_port: "data"
    output_port: "data"
  - node: "env_config"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT o.customer_id,\n       COUNT(DISTINCT o.order_id) AS orders_6m,\n       ROUND(SUM(CAST(l.quantity AS DECIMAL(10,0)) * CAST(COALESCE(l.unit_price, p.list_price) AS DECIMAL(12,2)) * (1 - CAST(l.discount_pct AS DECIMAL(5,2)))), 2) AS revenue_6m,\n       MAX(o.order_date) AS last_order_date\nFROM src_orders o\nJOIN src_order_lines l ON l.order_id = o.order_id\nLEFT JOIN src_products p ON p.product_id = l.product_id\nCROSS JOIN env_config v\nWHERE o.status = 'COMPLETED'\n  AND o.order_date >= CAST(v.min_order_date AS DATE)\nGROUP BY o.customer_id"
}
inputs = {
    "data": [
        ctx["src_orders.data"],
        ctx["src_order_lines.data"],
        ctx["src_products.data"],
        ctx["env_config.result"]
    ],
    "data__sources": [
        {
            "node": "src_orders",
            "output_port": "data",
            "name": "src_orders",
            "df_name": "src_orders"
        },
        {
            "node": "src_order_lines",
            "output_port": "data",
            "name": "src_order_lines",
            "df_name": "src_order_lines"
        },
        {
            "node": "src_products",
            "output_port": "data",
            "name": "src_products",
            "df_name": "src_products"
        },
        {
            "node": "env_config",
            "output_port": "result",
            "name": "env_config",
            "df_name": "env_config"
        }
    ]
}
out = run(config, inputs, spark)
ctx["recent_orders.result"] = out["result"]


In [0]:
"""
id: "sess_gaps"
template: "sql"
templateVersion: "1.0.0"
name: "sess_gaps"
position:
  x: 300
  y: 515
description:
  text: "Python recipe, part 1: new session when the gap to the previous event exceeds session_gap_minutes."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT e.*,
           CASE WHEN LAG(e.event_ts) OVER (PARTITION BY e.customer_id ORDER BY e.event_ts, e.event_id) IS NULL
                  OR (UNIX_TIMESTAMP(e.event_ts) - UNIX_TIMESTAMP(LAG(e.event_ts) OVER (PARTITION BY e.customer_id ORDER BY e.event_ts, e.event_id))) > v.session_gap_minutes * 60
                THEN 1 ELSE 0 END AS new_session
    FROM src_web_events e
    CROSS JOIN env_config v
input:
  - node: "src_web_events"
    input_port: "data"
    output_port: "data"
  - node: "env_config"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT e.*,\n       CASE WHEN LAG(e.event_ts) OVER (PARTITION BY e.customer_id ORDER BY e.event_ts, e.event_id) IS NULL\n              OR (UNIX_TIMESTAMP(e.event_ts) - UNIX_TIMESTAMP(LAG(e.event_ts) OVER (PARTITION BY e.customer_id ORDER BY e.event_ts, e.event_id))) > v.session_gap_minutes * 60\n            THEN 1 ELSE 0 END AS new_session\nFROM src_web_events e\nCROSS JOIN env_config v"
}
inputs = {
    "data": [
        ctx["src_web_events.data"],
        ctx["env_config.result"]
    ],
    "data__sources": [
        {
            "node": "src_web_events",
            "output_port": "data",
            "name": "src_web_events",
            "df_name": "src_web_events"
        },
        {
            "node": "env_config",
            "output_port": "result",
            "name": "env_config",
            "df_name": "env_config"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sess_gaps.result"] = out["result"]


In [0]:
"""
id: "sess_numbered"
template: "sql"
templateVersion: "1.0.0"
name: "sess_numbered"
position:
  x: 600
  y: 360
description:
  text: "Python recipe, part 2: session_no = cumsum(new_session) per customer."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT *, SUM(new_session) OVER (PARTITION BY customer_id ORDER BY event_ts, event_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_no
    FROM sess_gaps
input:
  - node: "sess_gaps"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *, SUM(new_session) OVER (PARTITION BY customer_id ORDER BY event_ts, event_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_no\nFROM sess_gaps"
}
inputs = {
    "data": [
        ctx["sess_gaps.result"]
    ],
    "data__sources": [
        {
            "node": "sess_gaps",
            "output_port": "result",
            "name": "sess_gaps",
            "df_name": "sess_gaps"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sess_numbered.result"] = out["result"]


In [0]:
"""
id: "sess_flags"
template: "transform"
templateVersion: "1.0.0"
name: "sess_flags"
position:
  x: 900
  y: 360
description:
  text: "Python recipe, part 3: checkout flag."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "CASE WHEN event_type = 'checkout' THEN 1 ELSE 0 END AS is_checkout"
input:
  - node: "sess_numbered"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "CASE WHEN event_type = 'checkout' THEN 1 ELSE 0 END AS is_checkout"
    ]
}
inputs = {
    "data": ctx["sess_numbered.result"]
}
out = run(config, inputs, spark)
ctx["sess_flags.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "sessions"
template: "aggregate"
templateVersion: "1.0.0"
name: "sessions"
position:
  x: 1200
  y: 360
description:
  text: "Python recipe, part 4: per-session aggregates."
  hash: ""
previewMode: "1000"
config:
  group_bys:
    - expr: "customer_id"
      type: "expr"
    - expr: "session_no"
      type: "expr"
  aggregations:
    - columnExpr:
        expr: "event_ts"
        type: "expr"
      fn: "MIN"
      alias: "session_start"
    - columnExpr:
        expr: "event_ts"
        type: "expr"
      fn: "MAX"
      alias: "session_end"
    - columnExpr:
        expr: "event_id"
        type: "expr"
      fn: "COUNT"
      alias: "n_events"
    - columnExpr:
        expr: "is_checkout"
        type: "expr"
      fn: "MAX"
      alias: "has_checkout"
input:
  - node: "sess_flags"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

# Default percentile (median). Used as a fallback when a PERCENTILE aggregation
# is missing or has a non-numeric / non-finite percentage. Mirrors
# DEFAULT_PERCENTILE_VALUE in AggregateNodeViewModel.ts so SQL and Python
# pipelines agree on the default.
DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        elif fn == "PERCENTILE":
            # Spark PERCENTILE requires (col, percentage [, frequency]) with percentage
            # in [0, 1]; emitting PERCENTILE(col) triggers WRONG_NUM_ARGS at analysis
            # time. For finite numeric inputs we clamp to [0, 1] (mirroring the V1/V2
            # input clamps) so out-of-range values like -0.1 or 1.5 — typos or stale
            # frontend state — become a valid boundary instead of silently overriding
            # the user's intent with the median. Non-numeric, non-finite, boolean, or
            # missing values fall back to DEFAULT_PERCENTILE.
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_id",
            "type": "expr"
        },
        {
            "expr": "session_no",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "event_ts",
                "type": "expr"
            },
            "fn": "MIN",
            "alias": "session_start"
        },
        {
            "columnExpr": {
                "expr": "event_ts",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "session_end"
        },
        {
            "columnExpr": {
                "expr": "event_id",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "n_events"
        },
        {
            "columnExpr": {
                "expr": "is_checkout",
                "type": "expr"
            },
            "fn": "MAX",
            "alias": "has_checkout"
        }
    ]
}
inputs = {
    "data": ctx["sess_flags.transformed_data"]
}
out = run(config, inputs, spark)
ctx["sessions.aggregated_data"] = out["aggregated_data"]


In [0]:
"""
id: "sessions_duration"
template: "transform"
templateVersion: "1.0.0"
name: "sessions_duration"
position:
  x: 1500
  y: 360
description:
  text: "Python recipe, part 5: duration_min."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "(UNIX_TIMESTAMP(session_end) - UNIX_TIMESTAMP(session_start)) / 60.0 AS duration_min"
input:
  - node: "sessions"
    input_port: "data"
    output_port: "aggregated_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "(UNIX_TIMESTAMP(session_end) - UNIX_TIMESTAMP(session_start)) / 60.0 AS duration_min"
    ]
}
inputs = {
    "data": ctx["sessions.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sessions_duration.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "sessions_by_customer"
template: "aggregate"
templateVersion: "1.0.0"
name: "sessions_by_customer"
position:
  x: 1800
  y: 360
description:
  text: "Group recipe compute_sessions_by_customer (globalCount = count)."
  hash: ""
previewMode: "1000"
config:
  group_bys:
    - expr: "customer_id"
      type: "expr"
  aggregations:
    - columnExpr:
        expr: "session_no"
        type: "expr"
      fn: "COUNT"
      alias: "count"
    - columnExpr:
        expr: "n_events"
        type: "expr"
      fn: "SUM"
      alias: "n_events_sum"
    - columnExpr:
        expr: "has_checkout"
        type: "expr"
      fn: "SUM"
      alias: "has_checkout_sum"
    - columnExpr:
        expr: "duration_min"
        type: "expr"
      fn: "AVG"
      alias: "duration_min_avg"
input:
  - node: "sessions_duration"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

# Default percentile (median). Used as a fallback when a PERCENTILE aggregation
# is missing or has a non-numeric / non-finite percentage. Mirrors
# DEFAULT_PERCENTILE_VALUE in AggregateNodeViewModel.ts so SQL and Python
# pipelines agree on the default.
DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        elif fn == "PERCENTILE":
            # Spark PERCENTILE requires (col, percentage [, frequency]) with percentage
            # in [0, 1]; emitting PERCENTILE(col) triggers WRONG_NUM_ARGS at analysis
            # time. For finite numeric inputs we clamp to [0, 1] (mirroring the V1/V2
            # input clamps) so out-of-range values like -0.1 or 1.5 — typos or stale
            # frontend state — become a valid boundary instead of silently overriding
            # the user's intent with the median. Non-numeric, non-finite, boolean, or
            # missing values fall back to DEFAULT_PERCENTILE.
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "customer_id",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "session_no",
                "type": "expr"
            },
            "fn": "COUNT",
            "alias": "count"
        },
        {
            "columnExpr": {
                "expr": "n_events",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "n_events_sum"
        },
        {
            "columnExpr": {
                "expr": "has_checkout",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "has_checkout_sum"
        },
        {
            "columnExpr": {
                "expr": "duration_min",
                "type": "expr"
            },
            "fn": "AVG",
            "alias": "duration_min_avg"
        }
    ]
}
inputs = {
    "data": ctx["sessions_duration.transformed_data"]
}
out = run(config, inputs, spark)
ctx["sessions_by_customer.aggregated_data"] = out["aggregated_data"]


In [0]:
"""
id: "event_pivot"
template: "pivot"
templateVersion: "1.0.0"
name: "event_pivot"
position:
  x: 300
  y: 670
description:
  text: "Pivot recipe: count per event_type per customer_id."
  hash: ""
previewMode: "1000"
config:
  mode: "pivot"
  pivot_column: "event_type"
  value_column: "event_id"
  agg_fn: "COUNT"
  null_behavior: "null"
  unpivot_columns: []
  key_column_name: "Column Name"
  value_column_name: "Value"
  exclude_columns:
    - expr: "event_ts"
      type: "column"
    - expr: "page"
      type: "column"
input:
  - node: "src_web_events"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List, Optional, Set
import pyspark.sql.functions as F

AGG_FUNCTIONS = {
    "SUM": F.sum,
    "AVG": F.avg,
    "COUNT": F.count,
    "MIN": F.min,
    "MAX": F.max,
}

def _validate_columns_exist(df, columns: List[str], label: str) -> None:
    df_cols: Set[str] = set(df.columns)
    missing = [c for c in columns if c not in df_cols]
    if missing:
        raise ValueError(
            f"{label}: column(s) not found in input data: {missing}. "
            f"Available columns: {df.columns}"
        )

def _validate_aggregation(agg_fn: str) -> None:
    if agg_fn not in AGG_FUNCTIONS:
        raise ValueError(
            f"Unsupported aggregation '{agg_fn}'. "
            f"Choose one of: {list(AGG_FUNCTIONS.keys())}"
        )

def _validate_no_overlap(pivot_cols: List[str], fixed_cols: List[str]) -> None:
    overlap = set(pivot_cols) & set(fixed_cols)
    if overlap:
        raise ValueError(
            f"Column(s) {sorted(overlap)} appear in both pivot and fixed roles. "
            "A column cannot serve both roles."
        )

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    mode: str = config.get("mode", "pivot")
    exclude_cfg: List[Dict[str, str]] = config.get("exclude_columns", [])
    excluded: Set[str] = {c.get("expr", "") for c in exclude_cfg if c.get("expr")}

    if mode == "unpivot":
        unpivot_cols_cfg: List[Dict[str, str]] = config.get("unpivot_columns", [])
        key_col: str = config.get("key_column_name", "Column Name")
        value_col: str = config.get("value_column_name", "Value")

        unpivot_col_names = [c.get("expr", "") for c in unpivot_cols_cfg if c.get("expr")]
        if not unpivot_col_names:
            raise ValueError("At least one column must be selected for unpivoting.")

        _validate_columns_exist(df, unpivot_col_names, "Unpivot columns")
        if excluded:
            _validate_columns_exist(df, list(excluded), "Exclude columns")

        fixed_cols = [
            c for c in df.columns
            if c not in unpivot_col_names and c not in excluded
        ]
        _validate_no_overlap(unpivot_col_names, fixed_cols)

        n = len(unpivot_col_names)
        stack_parts = ", ".join(
            [f"'{col}', `{col}`" for col in unpivot_col_names]
        )
        stack_expr = (
            f"stack({n}, {stack_parts}) "
            f"as (`{key_col}`, `{value_col}`)"
        )
        result = df.select(
            *[F.col(c) for c in fixed_cols],
            F.expr(stack_expr),
        )
        return {"pivoted_data": result}

    # ── Pivot (Rows → Columns) ───────────────────────────────────────────
    pivot_column: str = config.get("pivot_column", "")
    value_column: str = config.get("value_column", "")
    agg_fn: str = config.get("agg_fn", "SUM")
    null_behavior: str = config.get("null_behavior", "null")

    if not pivot_column or not value_column:
        return {"pivoted_data": df}

    _validate_columns_exist(df, [pivot_column, value_column], "Pivot")
    if excluded:
        _validate_columns_exist(df, list(excluded), "Exclude columns")
    _validate_aggregation(agg_fn)

    fixed_cols = [
        c for c in df.columns
        if c not in (pivot_column, value_column) and c not in excluded
    ]
    agg_func = AGG_FUNCTIONS[agg_fn]

    result = (
        df
        .select(*fixed_cols, pivot_column, value_column)
        .groupBy(fixed_cols)
        .pivot(pivot_column)
        .agg(agg_func(F.col(value_column)))
    )
    if null_behavior == "zero":
        result = result.fillna(0)
    return {"pivoted_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "pivot",
    "pivot_column": "event_type",
    "value_column": "event_id",
    "agg_fn": "COUNT",
    "null_behavior": "null",
    "unpivot_columns": [],
    "key_column_name": "Column Name",
    "value_column_name": "Value",
    "exclude_columns": [
        {
            "expr": "event_ts",
            "type": "column"
        },
        {
            "expr": "page",
            "type": "column"
        }
    ]
}
inputs = {
    "data": ctx["src_web_events.data"]
}
out = run(config, inputs, spark)
ctx["event_pivot.pivoted_data"] = out["pivoted_data"]


In [0]:
"""
id: "event_pivot_names"
template: "transform"
templateVersion: "1.0.0"
name: "event_pivot_names"
position:
  x: 600
  y: 515
description:
  text: "DSS pivot count naming {value}_count."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "`add_to_cart` AS add_to_cart_count"
    - "`checkout` AS checkout_count"
    - "`view` AS view_count"
input:
  - node: "event_pivot"
    input_port: "data"
    output_port: "pivoted_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "`add_to_cart` AS add_to_cart_count",
        "`checkout` AS checkout_count",
        "`view` AS view_count"
    ]
}
inputs = {
    "data": ctx["event_pivot.pivoted_data"]
}
out = run(config, inputs, spark)
ctx["event_pivot_names.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "customers_prepared"
template: "transform"
templateVersion: "1.0.0"
name: "customers_prepared"
position:
  x: 300
  y: 825
description:
  text: "Prepare: TRIM/UPPER country, FindReplace -> country_name (unmatched keep value), TRIM/UPPER/FillEmpty segment, ColumnsSelector."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "UPPER(TRIM(country)) AS country"
    - "CASE UPPER(TRIM(country)) WHEN 'DE' THEN 'Germany' WHEN 'AT' THEN 'Austria' WHEN 'CH' THEN 'Switzerland' WHEN 'NL' THEN 'Netherlands' WHEN 'FR' THEN 'France' ELSE UPPER(TRIM(country)) END AS country_name"
    - "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment"
    - "status"
input:
  - node: "src_customers"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "UPPER(TRIM(country)) AS country",
        "CASE UPPER(TRIM(country)) WHEN 'DE' THEN 'Germany' WHEN 'AT' THEN 'Austria' WHEN 'CH' THEN 'Switzerland' WHEN 'NL' THEN 'Netherlands' WHEN 'FR' THEN 'France' ELSE UPPER(TRIM(country)) END AS country_name",
        "CASE WHEN segment IS NULL OR TRIM(segment) = '' THEN 'UNKNOWN' ELSE UPPER(TRIM(segment)) END AS segment",
        "status"
    ]
}
inputs = {
    "data": ctx["src_customers.data"]
}
out = run(config, inputs, spark)
ctx["customers_prepared.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "customers_drop_x"
template: "filter"
templateVersion: "1.0.0"
name: "customers_drop_x"
position:
  x: 600
  y: 670
description:
  text: "Prepare: FilterOnValue status = X (REMOVE_ROW)."
  hash: ""
previewMode: "1000"
config:
  condition: "status IS NULL OR status <> 'X'"
input:
  - node: "customers_prepared"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status IS NULL OR status <> 'X'"
}
inputs = {
    "data": ctx["customers_prepared.transformed_data"]
}
out = run(config, inputs, spark)
ctx["customers_drop_x.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "j_orders"
template: "join"
templateVersion: "1.0.0"
name: "j_orders"
position:
  x: 900
  y: 515
description:
  text: "Join LEFT recent_orders."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`customer_id` = right.`customer_id`"
  expressions:
    - "left.*"
    - "right.`orders_6m` AS `orders_6m`"
    - "right.`revenue_6m` AS `revenue_6m`"
    - "right.`last_order_date` AS `last_order_date`"
input:
  - node: "customers_drop_x"
    input_port: "left"
    output_port: "filtered_data"
  - node: "recent_orders"
    input_port: "right"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`customer_id` = right.`customer_id`",
    "expressions": [
        "left.*",
        "right.`orders_6m` AS `orders_6m`",
        "right.`revenue_6m` AS `revenue_6m`",
        "right.`last_order_date` AS `last_order_date`"
    ]
}
inputs = {
    "left": ctx["customers_drop_x.filtered_data"],
    "right": ctx["recent_orders.result"]
}
out = run(config, inputs, spark)
ctx["j_orders.joined_data"] = out["joined_data"]


In [0]:
"""
id: "j_sessions"
template: "join"
templateVersion: "1.0.0"
name: "j_sessions"
position:
  x: 2100
  y: 360
description:
  text: "Join LEFT sessions_by_customer (with aliases)."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`customer_id` = right.`customer_id`"
  expressions:
    - "left.*"
    - "right.`count` AS `sessions`"
    - "right.`n_events_sum` AS `total_events`"
    - "right.`has_checkout_sum` AS `checkout_sessions`"
    - "right.`duration_min_avg` AS `avg_session_minutes`"
input:
  - node: "j_orders"
    input_port: "left"
    output_port: "joined_data"
  - node: "sessions_by_customer"
    input_port: "right"
    output_port: "aggregated_data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`customer_id` = right.`customer_id`",
    "expressions": [
        "left.*",
        "right.`count` AS `sessions`",
        "right.`n_events_sum` AS `total_events`",
        "right.`has_checkout_sum` AS `checkout_sessions`",
        "right.`duration_min_avg` AS `avg_session_minutes`"
    ]
}
inputs = {
    "left": ctx["j_orders.joined_data"],
    "right": ctx["sessions_by_customer.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["j_sessions.joined_data"] = out["joined_data"]


In [0]:
"""
id: "j_events"
template: "join"
templateVersion: "1.0.0"
name: "j_events"
position:
  x: 2400
  y: 360
description:
  text: "Join LEFT event_pivot."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`customer_id` = right.`customer_id`"
  expressions:
    - "left.*"
    - "right.`add_to_cart_count` AS `add_to_cart_count`"
    - "right.`checkout_count` AS `checkout_count`"
    - "right.`view_count` AS `view_count`"
input:
  - node: "j_sessions"
    input_port: "left"
    output_port: "joined_data"
  - node: "event_pivot_names"
    input_port: "right"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`customer_id` = right.`customer_id`",
    "expressions": [
        "left.*",
        "right.`add_to_cart_count` AS `add_to_cart_count`",
        "right.`checkout_count` AS `checkout_count`",
        "right.`view_count` AS `view_count`"
    ]
}
inputs = {
    "left": ctx["j_sessions.joined_data"],
    "right": ctx["event_pivot_names.transformed_data"]
}
out = run(config, inputs, spark)
ctx["j_events.joined_data"] = out["joined_data"]


In [0]:
"""
id: "j_env"
template: "join"
templateVersion: "1.0.0"
name: "j_env"
position:
  x: 2700
  y: 360
description:
  text: "Attach ${analysis_end} for the formulas."
  hash: ""
previewMode: "1000"
config:
  join_type: "cross"
  join_conditions: ""
  expressions:
    - "left.*"
    - "right.`analysis_end` AS `analysis_end`"
input:
  - node: "j_events"
    input_port: "left"
    output_port: "joined_data"
  - node: "env_config"
    input_port: "right"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "cross",
    "join_conditions": "",
    "expressions": [
        "left.*",
        "right.`analysis_end` AS `analysis_end`"
    ]
}
inputs = {
    "left": ctx["j_events.joined_data"],
    "right": ctx["env_config.result"]
}
out = run(config, inputs, spark)
ctx["j_env.joined_data"] = out["joined_data"]


In [0]:
"""
id: "scored"
template: "transform"
templateVersion: "1.0.0"
name: "scored"
position:
  x: 3000
  y: 360
description:
  text: "Prepare compute_customer_360_scored: FillEmpty 0, days_since_last_order, conversion_rate, engagement_tier, Round avg_session_minutes (1)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "customer_id"
    - "country"
    - "country_name"
    - "segment"
    - "status"
    - "COALESCE(orders_6m, 0) AS orders_6m"
    - "COALESCE(revenue_6m, 0) AS revenue_6m"
    - "last_order_date"
    - "COALESCE(sessions, 0) AS sessions"
    - "COALESCE(total_events, 0) AS total_events"
    - "COALESCE(checkout_sessions, 0) AS checkout_sessions"
    - "ROUND(avg_session_minutes, 1) AS avg_session_minutes"
    - "COALESCE(add_to_cart_count, 0) AS add_to_cart_count"
    - "COALESCE(checkout_count, 0) AS checkout_count"
    - "COALESCE(view_count, 0) AS view_count"
    - "CASE WHEN last_order_date IS NULL THEN -1 ELSE DATEDIFF(CAST(analysis_end AS DATE), last_order_date) END AS days_since_last_order"
    - "CASE WHEN COALESCE(view_count, 0) > 0 THEN FLOOR(COALESCE(checkout_count, 0) / view_count * 10000 + 0.5) / 10000 ELSE 0 END AS conversion_rate"
    - "CASE WHEN COALESCE(revenue_6m, 0) >= 500 AND COALESCE(sessions, 0) >= 3 THEN 'HIGH' WHEN COALESCE(revenue_6m, 0) >= 100 OR COALESCE(sessions, 0) >= 1 THEN 'MEDIUM' ELSE 'LOW' END AS engagement_tier"
input:
  - node: "j_env"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "customer_id",
        "country",
        "country_name",
        "segment",
        "status",
        "COALESCE(orders_6m, 0) AS orders_6m",
        "COALESCE(revenue_6m, 0) AS revenue_6m",
        "last_order_date",
        "COALESCE(sessions, 0) AS sessions",
        "COALESCE(total_events, 0) AS total_events",
        "COALESCE(checkout_sessions, 0) AS checkout_sessions",
        "ROUND(avg_session_minutes, 1) AS avg_session_minutes",
        "COALESCE(add_to_cart_count, 0) AS add_to_cart_count",
        "COALESCE(checkout_count, 0) AS checkout_count",
        "COALESCE(view_count, 0) AS view_count",
        "CASE WHEN last_order_date IS NULL THEN -1 ELSE DATEDIFF(CAST(analysis_end AS DATE), last_order_date) END AS days_since_last_order",
        "CASE WHEN COALESCE(view_count, 0) > 0 THEN FLOOR(COALESCE(checkout_count, 0) / view_count * 10000 + 0.5) / 10000 ELSE 0 END AS conversion_rate",
        "CASE WHEN COALESCE(revenue_6m, 0) >= 500 AND COALESCE(sessions, 0) >= 3 THEN 'HIGH' WHEN COALESCE(revenue_6m, 0) >= 100 OR COALESCE(sessions, 0) >= 1 THEN 'MEDIUM' ELSE 'LOW' END AS engagement_tier"
    ]
}
inputs = {
    "data": ctx["j_env.joined_data"]
}
out = run(config, inputs, spark)
ctx["scored.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "rank_in_country"
template: "sql"
templateVersion: "1.0.0"
name: "rank_in_country"
position:
  x: 3300
  y: 360
description:
  text: "Window recipe compute_customer_360: rank by revenue_6m desc within country."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT *, RANK() OVER (PARTITION BY country ORDER BY revenue_6m DESC) AS `rank`
    FROM scored
input:
  - node: "scored"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *, RANK() OVER (PARTITION BY country ORDER BY revenue_6m DESC) AS `rank`\nFROM scored"
}
inputs = {
    "data": [
        ctx["scored.transformed_data"]
    ],
    "data__sources": [
        {
            "node": "scored",
            "output_port": "transformed_data",
            "name": "scored",
            "df_name": "scored"
        }
    ]
}
out = run(config, inputs, spark)
ctx["rank_in_country.result"] = out["result"]


In [0]:
"""
id: "out_customer_360"
template: "output"
templateVersion: "1.0.0"
name: "out_customer_360"
position:
  x: 3600
  y: 360
description:
  text: "Managed dataset customer_360."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "dss_05_customer_360"
input:
  - node: "rank_in_country"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "dss_05_customer_360"
}
inputs = {
    "data": ctx["rank_in_country.result"]
}
out = run(config, inputs, spark)
